# Task 6: Off-the-shelf tools vs. my model

**Objective:** try publicly available tools (Claude, ChatGPT, Gemini, Teachable Machine) on the
real-vs-fake problem, report how accurate they are and what makes them hard to use, and compare
them with my own model.

Adam Oakes

**How the comparison works**
* **One fixed sample for everyone:** 200 images from the *test* set (100 REAL, 100 FAKE, chosen at
  random with a fixed seed), shuffled and renamed `img_001` … `img_200` so the file names give
  nothing away. The answer key is in `results/task_6/answer_key.csv`.
* **For the chat tools,** the images are enlarged 8× (32×32 → 256×256, nearest-neighbor, so no
  pixels are blended) and arranged on 8 numbered contact sheets in `data/task_6_sample/sheets/`.
  Each tool gets the same sheets and the same prompt (`data/task_6_sample/PROMPT.txt`), with no
  examples and no information about how many images are fake.
* **Each tool's answers** go in `results/task_6/predictions/<tool>.csv` (columns `id,pred`).
  This notebook scores every file it finds there.
* **My model** is the best one from Task 5: ImageNet-pre-trained ResNet50, fine-tuned, with
  horizontal flip and Dropout(0.5). It is scored on the same 200 images and also on the full
  20,000-image test set, which no earlier task used.

**Metrics:** accuracy, plus precision, recall and F1 with **FAKE as the positive class**, and a
confusion matrix. Accuracy alone hides *how* a tool fails: a tool that calls almost everything
REAL can still score well above 50%. With only 200 images, accuracy is uncertain by about ±6
points, so each accuracy is shown with a 95% confidence interval.

# Library imports and configuration

In [ ]:
import os, re, csv, json, time
from pathlib import Path

import keras
from keras import layers
from keras.utils import image_dataset_from_directory
import tensorflow as tf
for _gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_gpu, True)
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
plt.style.use("default")

def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "train").is_dir():
            return d
    raise FileNotFoundError(f"Couldn't find the project folder (one that contains data/train) above {here}.")

ROOT = Path(os.environ["PROJECT_ROOT"]) if "PROJECT_ROOT" in os.environ else find_project_root()
DATA_DIR = ROOT / "data"
results_dir = ROOT / "results" / "task_6"
pred_dir = results_dir / "predictions"
plots = ROOT / "plots" / "task_6"
ckpt_dir = ROOT / "checkpoints" / "task_6"
for d in (pred_dir, plots, ckpt_dir):
    d.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
NATIVE_SIZE = (32, 32)
MODEL_SIZE = (180, 180)
SEED = 42
FE_EPOCHS = int(os.environ.get("FE_EPOCHS", 10))
FT_EPOCHS = int(os.environ.get("FT_EPOCHS", 10))
FT_LR = 1e-5
PRETRAINED_WEIGHTS = os.environ.get("PRETRAINED_WEIGHTS", "imagenet")

KEY = pd.read_csv(results_dir / "answer_key.csv")          # id, label, source
print("GPUs:", len(tf.config.list_physical_devices("GPU")), "| project:", ROOT)
print("sample:", KEY.label.value_counts().to_dict())

# My model: fine-tuned ResNet50 (Task 5's best)

Task 5 saved only the phase-1 (frozen-base) weights, so the fine-tuning phase is repeated here
from those weights, with the same settings, and the result is saved to `checkpoints/task_6/`.
That takes roughly 10 minutes on the GPU the first time; later runs load the saved weights.
If the Task 5 phase-1 weights aren't on this machine, phase 1 is trained here first.

In [ ]:
def build_resnet50():
    base = keras.applications.ResNet50(weights=PRETRAINED_WEIGHTS, include_top=False,
                                       input_shape=MODEL_SIZE + (3,))
    inputs = keras.Input(shape=NATIVE_SIZE + (3,))
    x = layers.RandomFlip("horizontal")(inputs)                       # Task 3
    x = layers.Resizing(*MODEL_SIZE, interpolation="bilinear")(x)
    x = layers.Lambda(keras.applications.resnet50.preprocess_input, name="preprocess")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)                                        # Task 4
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="resnet50_imagenet"), base

def set_trainable(base, mode):
    base.trainable = mode != "none"
    if mode == "top":
        for layer in base.layers:
            layer.trainable = layer.name.startswith("conv5_")

def fit(model, epochs, lr, train_ds, val_ds):
    model.compile(loss="binary_crossentropy", metrics=["accuracy"],
                  optimizer=keras.optimizers.Adam(learning_rate=lr))
    cb = [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, mode="max",
                                        restore_best_weights=True)]
    return model.fit(train_ds, epochs=epochs, validation_data=val_ds, callbacks=cb, verbose=2)

FT_WEIGHTS = ckpt_dir / "resnet50_imagenet_fine_tune.weights.h5"
FE_WEIGHTS = ROOT / "checkpoints" / "task_5" / "resnet50_imagenet_feature_extract_flip_drop05_e20.weights.h5"

keras.utils.set_random_seed(SEED)
model, base = build_resnet50()
set_trainable(base, "none")                       # phase-1 layout

if FT_WEIGHTS.exists():
    set_trainable(base, "top")
    model.load_weights(FT_WEIGHTS)
    print("loaded fine-tuned weights:", FT_WEIGHTS.name)
else:
    def load(split, shuffle):
        return image_dataset_from_directory(DATA_DIR / split, image_size=NATIVE_SIZE,
                                            batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED)
    train_ds = (load("train", True).unbatch().cache()
                .shuffle(20_000, seed=SEED, reshuffle_each_iteration=True)
                .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
    val_ds = load("validation", False).cache().prefetch(tf.data.AUTOTUNE)
    if FE_WEIGHTS.exists():
        model.load_weights(FE_WEIGHTS)
        print("loaded Task 5 phase-1 weights")
    else:
        print("Task 5 phase-1 weights not found; training phase 1 (frozen base) here")
        fit(model, FE_EPOCHS, 1e-3, train_ds, val_ds)
    set_trainable(base, "top")
    h = fit(model, FT_EPOCHS, FT_LR, train_ds, val_ds)
    model.save_weights(FT_WEIGHTS)
    print(f"fine-tuned; best val_acc {max(h.history['val_accuracy']):.4f}")

## Score my model

On the full test set (20,000 images, never used before this task) and on the 200-image sample.
The model outputs the probability of REAL (class index 1), so an image is called FAKE when
that probability is below 0.5.

In [ ]:
test_ds = image_dataset_from_directory(DATA_DIR / "test", image_size=NATIVE_SIZE,
                                       batch_size=BATCH_SIZE, shuffle=False)
assert test_ds.class_names == ["FAKE", "REAL"]
y_true_test = np.concatenate([y.numpy() for _, y in test_ds])
t0 = time.time()
p_test = model.predict(test_ds, verbose=0).ravel()
test_sec = time.time() - t0
test_true = np.where(y_true_test == 1, "REAL", "FAKE")
test_pred = np.where(p_test >= 0.5, "REAL", "FAKE")

sample_x = np.stack([np.asarray(Image.open(ROOT / s).convert("RGB").resize(NATIVE_SIZE), dtype="float32")
                     for s in KEY.source])
t0 = time.time()
p_sample = model.predict(sample_x, batch_size=BATCH_SIZE, verbose=0).ravel()
sample_sec = time.time() - t0
pd.DataFrame({"id": KEY.id, "pred": np.where(p_sample >= 0.5, "REAL", "FAKE")}).to_csv(
    pred_dir / "my_model_resnet50.csv", index=False)
print(f"test set: {len(test_true):,} images in {test_sec:.1f} s | sample: 200 images in {sample_sec:.2f} s")

# Add a tool's answers

Run the tool on the 8 contact sheets with the prompt in `data/task_6_sample/PROMPT.txt`, then
paste its replies (all sheets, one after another) into `save_predictions`. It picks out every
line that has an image number followed by REAL or FAKE, so extra words in the reply don't matter.
A number that never appears is counted as unanswered, and unanswered counts as wrong.

`claude.csv` is already there: Claude classified the sample from the same sheets, without the key.

In [ ]:
def save_predictions(tool, text):
    found = {}
    for num, lab in re.findall(r"(?<!\d)(\d{1,3})\D{0,12}?\b(REAL|FAKE)\b", text.upper()):
        if 1 <= int(num) <= len(KEY):
            found[int(num)] = lab
    out = pred_dir / f"{tool}.csv"
    pd.DataFrame({"id": sorted(found), "pred": [found[i] for i in sorted(found)]}).to_csv(out, index=False)
    print(f"{tool}: saved {len(found)} of {len(KEY)} answers -> {out.name}")

# Example (delete the leading '#' marks, paste the replies between the triple quotes, and run):
# save_predictions("chatgpt", '''
# 1,REAL
# 2,FAKE
# ...
# ''')

# Results

In [ ]:
def metrics(true, pred):
    true, pred = np.asarray(true), np.asarray(pred)
    tp = int(((true == "FAKE") & (pred == "FAKE")).sum()); fn = int(((true == "FAKE") & (pred == "REAL")).sum())
    n_fake = int((true == "FAKE").sum())
    fp = int(((true == "REAL") & (pred == "FAKE")).sum()); tn = int(((true == "REAL") & (pred == "REAL")).sum())
    n = len(true); acc = (tp + tn) / n
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / n_fake if n_fake else float("nan")       # an unanswered FAKE image counts as missed
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else float("nan")
    z = 1.96                                             # 95% Wilson interval for accuracy
    centre = (acc + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(acc * (1 - acc) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return dict(n=n, n_fake=n_fake, accuracy=acc, acc_low=centre - half, acc_high=centre + half,
                precision_fake=prec, recall_fake=rec, f1_fake=f1,
                fake_called_fake=tp, fake_called_real=fn, real_called_fake=fp, real_called_real=tn)

rows = [dict(tool="my_model_resnet50 (full test set)", unanswered=0, **metrics(test_true, test_pred))]
truth = KEY.set_index("id").label
for f in sorted(pred_dir.glob("*.csv")):
    p = pd.read_csv(f).drop_duplicates("id").set_index("id").pred.astype(str).str.upper().str.strip()
    pred = p.reindex(truth.index).fillna("UNANSWERED")
    rows.append(dict(tool=f.stem, unanswered=int((pred == "UNANSWERED").sum()), **metrics(truth.values, pred.values)))
res = pd.DataFrame(rows)
res.to_csv(results_dir / "tool_comparison.csv", index=False)

def to_markdown(df):
    f3 = "{:.3f}".format
    lines = ["| tool | images | accuracy (95% CI) | precision (FAKE) | recall (FAKE) | F1 (FAKE) | unanswered |",
             "|---|---|---|---|---|---|---|"]
    for _, r in df.iterrows():
        lines.append(f"| {r.tool} | {r.n:,} | {r.accuracy:.3f} ({r.acc_low:.3f}–{r.acc_high:.3f}) | "
                     f"{f3(r.precision_fake)} | {f3(r.recall_fake)} | {f3(r.f1_fake)} | {r.unanswered} |")
    return "\n".join(lines)

md_table = to_markdown(res)
(results_dir / "tool_comparison.md").write_text(md_table)
from IPython.display import Markdown, display
display(Markdown(md_table))

In [ ]:
# Accuracy with 95% confidence intervals
t = res.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 0.6 * len(t) + 1.4))
colors = ["#2a6fdb" if s.startswith("my_model") else "#8a8a8a" for s in t.tool]
ax.barh(t.tool, t.accuracy, color=colors, height=0.6,
        xerr=[t.accuracy - t.acc_low, t.acc_high - t.accuracy], ecolor="#333", capsize=3)
ax.axvline(0.5, color="#444", lw=1, ls="--")
for y, (hi, a) in enumerate(zip(t.acc_high, t.accuracy)):
    ax.text(hi + 0.01, y, f"{a:.3f}", va="center", fontsize=9, color="#333")
ax.set_xlim(0, 1.1); ax.set_xlabel("accuracy (bars: 95% confidence interval)")
ax.set_title("Accuracy by tool\nblue = my model, gray = off-the-shelf tools, dashed = guessing", fontsize=10)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); fig.savefig(plots / "accuracy_by_tool.png", dpi=150); plt.show()

# Confusion matrices (rows: truth, columns: prediction), as a share of each true class
fig, axes = plt.subplots(1, len(res), figsize=(3.4 * len(res), 3.6))
for ax, (_, r) in zip(np.atleast_1d(axes), res.iterrows()):
    m = np.array([[r.fake_called_fake, r.fake_called_real], [r.real_called_fake, r.real_called_real]], dtype=float)
    share = m / np.array([[max(r.n_fake, 1)], [max(r.n - r.n_fake, 1)]])   # share of all images in the true class
    ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{int(m[i, j]):,}\n{share[i, j]:.0%}", ha="center", va="center",
                    color="white" if share[i, j] > 0.55 else "#222", fontsize=10)
    ax.set_xticks([0, 1], ["called FAKE", "called REAL"]); ax.set_yticks([0, 1], ["is FAKE", "is REAL"])
    ax.set_title(r.tool + (f"\n({r.unanswered} unanswered)" if r.unanswered else ""), fontsize=9)
plt.tight_layout(); fig.savefig(plots / "confusion_matrices.png", dpi=150); plt.show()

# Discussion

*To be written once all the tools have been scored.*